In [1]:
%pip install "numpy<2"

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
print(np.__version__)

1.26.4


In [4]:
import pandas as pd
import numpy as np

np.random.seed(42) # Fixes the randomness so every time you run this code, you get the same "random" numbers. This makes results reproducible.

#Creates a list of weekly dates from Jan 2022 to Dec 2024.
#freq="W" means one date per week → roughly 156 weeks total.

dates = pd.date_range(
    start="2022-01-01",
    end="2024-12-31",
    freq="W"
)

# Different SKU demand patterns
sku_patterns = {
    "ITEM_001": 80,    # low demand
    "ITEM_002": 150,   # medium demand
    "ITEM_003": 300    # high demand
}

rows = []

#data generation loop

for sku, avg_sales in sku_patterns.items():

    for date in dates:

        # Generate realistic sales
        sales = (
            np.random.poisson(lam=avg_sales)   # Simulates natural demand variability — common in retail/inventory modeling
            +
            np.random.randint(-15, 15)         # Adds a small random noise of ±15 units
        )

        rows.append({
            "date": date,
            "sku": sku,
            "sales": max(0, sales)             # Ensures sales never go negative

        })

# Total rows = 3 SKUs × ~156 weeks = ~468 rows

df = pd.DataFrame(rows)

# Save dataset
df.to_csv("walmart_sales.csv", index=False)

print(df.head(10))
print("\nAverage sales by SKU:\n")

print(
    df.groupby("sku")["sales"].mean()
)

C:\Users\KIIT\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\KIIT\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


        date       sku  sales
0 2022-01-02  ITEM_001     72
1 2022-01-09  ITEM_001     98
2 2022-01-16  ITEM_001     78
3 2022-01-23  ITEM_001     62
4 2022-01-30  ITEM_001    102
5 2022-02-06  ITEM_001     77
6 2022-02-13  ITEM_001     93
7 2022-02-20  ITEM_001     62
8 2022-02-27  ITEM_001     67
9 2022-03-06  ITEM_001     84

Average sales by SKU:

sku
ITEM_001     80.133758
ITEM_002    149.337580
ITEM_003    300.331210
Name: sales, dtype: float64


In [5]:
import sqlite3

# Create database connection
conn = sqlite3.connect("intellistock.db")

# Store dataset into SQLite
df.to_sql("sales", conn, if_exists="replace", index=False)

print("Database created successfully!")

# Verify data
test = pd.read_sql("SELECT * FROM sales LIMIT 5", conn)

print(test)

# Close connection
conn.close()

Database created successfully!
                  date       sku  sales
0  2022-01-02 00:00:00  ITEM_001     72
1  2022-01-09 00:00:00  ITEM_001     98
2  2022-01-16 00:00:00  ITEM_001     78
3  2022-01-23 00:00:00  ITEM_001     62
4  2022-01-30 00:00:00  ITEM_001    102


In [6]:
%pip install chronos-forecasting


Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch
import pandas as pd
import sqlite3

def get_context(sku, n=52):

    # Connect to SQLite database
    conn = sqlite3.connect("intellistock.db")

    # Read data for selected SKU
    df = pd.read_sql(
        f"SELECT * FROM sales WHERE sku='{sku}'",
        conn
    )

    # Close connection
    conn.close()

    # Take last n sales values
    series = df.tail(n)["sales"].tolist()

    # Convert to tensor
    context = torch.tensor(series, dtype=torch.float32)

    return context    # Shape will be torch.Size([52]) — a 1D tensor with 52 values.

print("Context function ready!")

print(f"Sample context shape: {get_context('ITEM_001').shape}")

Context function ready!
Sample context shape: torch.Size([52])


In [8]:
import chronos
print("Chronos ready!")

Chronos ready!


In [9]:
%pip install huggingface_hub[hf_xet]

Note: you may need to restart the kernel to use updated packages.


In [10]:
import torch
from chronos import ChronosPipeline

pipe = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cpu",
    torch_dtype=torch.float32
)

print("Chronos model loaded!")

`torch_dtype` is deprecated! Use `dtype` instead!


Chronos model loaded!


In [11]:
import chronos
print(chronos.__version__)

# check what arguments predict() accepts
import inspect
print(inspect.signature(pipe.predict))

2.2.2
(inputs: Union[torch.Tensor, List[torch.Tensor]], prediction_length: Optional[int] = None, num_samples: Optional[int] = None, temperature: Optional[float] = None, top_k: Optional[int] = None, top_p: Optional[float] = None, limit_prediction_length: bool = False) -> torch.Tensor


In [12]:
import torch
import pandas as pd
import sqlite3

def get_context(sku, n=52):

    # Connect to SQLite database
    conn = sqlite3.connect("intellistock.db")

    # Read SKU data
    df = pd.read_sql(
        f"SELECT * FROM sales WHERE sku='{sku}'",
        conn
    )

    # Close connection
    conn.close()

    # Get last n sales values
    series = df.tail(n)["sales"].tolist()

    # Convert to tensor
    context = torch.tensor(series, dtype=torch.float32)

    return context


def predict_demand(sku, horizon=4):   #  how many weeks ahead to predict (default = 4 weeks)

    context = get_context(sku, n=52)   # Fetches the last 52 weeks of sales data as a tensor.
                                    # This is the input to the forecasting model.

    forecast = pipe.predict(
        inputs=[context],
        prediction_length=horizon,
        num_samples=20
    )

    median_demand = float(
        forecast[0].median(dim=0).values.sum()
    )

    return round(median_demand, 2)


# Test all SKUs
for sku in ["ITEM_001", "ITEM_002", "ITEM_003"]:

    demand = predict_demand(sku)

    print(f"{sku} → predicted 4-week demand: {demand} units")

ITEM_001 → predicted 4-week demand: 328.73 units
ITEM_002 → predicted 4-week demand: 581.39 units
ITEM_003 → predicted 4-week demand: 1185.28 units


In [13]:
%pip install langchain-community faiss-cpu sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [14]:
%pip install faiss-gpu

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [15]:
%pip install tf-keras

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: tf-keras in c:\users\kiit\anaconda3\lib\site-packages (2.21.0)



In [16]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [17]:
#  What This Cell Does
  #It builds a semantic search engine over a supply chain knowledge base using FAISS + HuggingFace embeddings. This is the RAG (Retrieval-Augmented Generation) component of the system.


from langchain_community.vectorstores import FAISS     # FAISS enables ultra-fast nearest-neighbour search — even at millions of documents.
from langchain_community.embeddings import HuggingFaceEmbeddings

# Supply chain knowledge base
docs = [
    "Copper prices forecast to rise 12% in Q3 due to Chile mine strike.",
    "Supplier A lead time increased from 3 to 6 weeks updated May 2025.",
    "Sea freight rates from Shanghai up 30% use air for urgent orders.",
    "Supplier B offers 8% volume discount above 500 units.",
    "Steel shortage expected in Q4 due to port strikes in South Korea.",
    "Supplier C has been blacklisted in winter months due to delays.",
    "Cotton prices stable no major disruptions expected this quarter.",
    "Warehouse storage costs rising 15% recommend reducing buffer stock.",
]

# Load embedding model
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Build FAISS index
vectorstore = FAISS.from_texts(docs, embedder)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})   # k=3 means — when queried, return the top 3 most relevant documents.

print("FAISS vector store ready!")
print(f"Total documents indexed: {len(docs)}")

C:\Users\KIIT\AppData\Local\Temp\ipykernel_9636\4184320171.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS     # FAISS enables ultra-fast nearest-neighbour search — even at millions of documents.
C:\Users\KIIT\AppData\Local\Temp\ipykernel_9636\4184320171.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(



FAISS vector store ready!
Total documents indexed: 8


In [18]:
# Test semantic search

query = "Which suppliers have delivery delays?"

results = retriever.invoke(query)

print("\nTop 3 Relevant Results:\n")

for i, doc in enumerate(results, 1):

    print(f"Result {i}:")
    print(doc.page_content)
    print()


Top 3 Relevant Results:

Result 1:
Supplier C has been blacklisted in winter months due to delays.

Result 2:
Supplier A lead time increased from 3 to 6 weeks updated May 2025.

Result 3:
Sea freight rates from Shanghai up 30% use air for urgent orders.



In [19]:
# Create reusable market research function

def market_research(query):   # Takes a natural language question as input

    results = retriever.invoke(query)    #  list of LangChain Document objects.

    print("\nTop Market Insights:\n")

    for i, doc in enumerate(results, 1):

        print(f"Result {i}:")
        print(doc.page_content)
        print()


# Test function
market_research("Which suppliers are risky?")


Top Market Insights:

Result 1:
Supplier B offers 8% volume discount above 500 units.

Result 2:
Supplier C has been blacklisted in winter months due to delays.

Result 3:
Sea freight rates from Shanghai up 30% use air for urgent orders.



In [20]:
%pip install beautifulsoup4 requests newspaper3k

Note: you may need to restart the kernel to use updated packages.


In [21]:
import requests
from bs4 import BeautifulSoup

# ==========================================
# Simple Working Scraper
# ==========================================

url = "https://news.ycombinator.com/"

response = requests.get(url)

soup = BeautifulSoup(
    response.text,
    "html.parser"
)

titles = soup.find_all("span", class_="titleline")

scraped_docs = []

print("\nScraped Headlines:\n")

for t in titles[:10]:

    text = t.get_text(strip=True)

    scraped_docs.append(text)

    print("-", text)


Scraped Headlines:

- The most spectacular rocket explosion since N1 just happened in Florida(arstechnica.com)
- Claude Opus 4.8(anthropic.com)
- Bricks and Minifigs Stole a Man's $200k Lego Collection(mybricklog.com)
- I made a million dollar product from my dorm room (2025)(winans.io)
- Cars collect a startling amount of data about you(bbc.com)
- Ten Basic Clouds(noaa.gov)
- Blue Origin's New Glenn blows up during static fire test(twitter.com/nasaspaceflight)
- The and Wonderful Evolution of the Waterproof Jacket(carryology.com)
- Italians and Dutch share the same gestural instinct for teaching(mpi.nl)
- Nitpicking the shell history scene in 'Tron: Legacy'(greenend.org.uk)


In [22]:
all_docs = docs + scraped_docs   # List Concatenation.Merges the hardcoded supply chain docs with the live scraped headlines and rebuilds the FAISS vector store with the combined knowledge base

vectorstore = FAISS.from_texts(
    all_docs,
    embedder
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Updated vector database ready!")
print(f"Total documents: {len(all_docs)}")

Updated vector database ready!
Total documents: 18


In [23]:
print(scraped_docs)
print(len(scraped_docs))

['The most spectacular rocket explosion since N1 just happened in Florida(arstechnica.com)', 'Claude Opus 4.8(anthropic.com)', "Bricks and Minifigs Stole a Man's $200k Lego Collection(mybricklog.com)", 'I made a million dollar product from my dorm room (2025)(winans.io)', 'Cars collect a startling amount of data about you(bbc.com)', 'Ten Basic Clouds(noaa.gov)', "Blue Origin's New Glenn blows up during static fire test(twitter.com/nasaspaceflight)", 'The and Wonderful Evolution of the Waterproof Jacket(carryology.com)', 'Italians and Dutch share the same gestural instinct for teaching(mpi.nl)', "Nitpicking the shell history scene in 'Tron: Legacy'(greenend.org.uk)"]
10


In [24]:
%pip install pulp

Note: you may need to restart the kernel to use updated packages.


In [25]:
# linear programming (LP) optimization model that calculates the ideal quantity to order — balancing demand, budget, storage, and cost constraints. This is the procurement brain of the syste
from pulp import *

def optimize_inventory(
    predicted_demand,
    unit_cost=50,
    storage_cost=2,
    budget=30000,
    storage_capacity=1000,
    min_order=100
):

    model = LpProblem(
        "Inventory_Optimization",
        LpMinimize
    )

    # Decision variables
    order_qty = LpVariable(
        "order_quantity",
        lowBound=min_order,
        cat="Integer"
    )

    shortage = LpVariable(
        "shortage",
        lowBound=0,
        cat="Integer"
    )

    # Objective function
    model += (
        unit_cost * order_qty
        +
        storage_cost * order_qty
        +
        1000 * shortage
    )

    # Demand satisfaction
    model += order_qty + shortage >= predicted_demand

    # Budget constraint
    model += unit_cost * order_qty <= budget

    # Storage constraint
    model += order_qty <= storage_capacity

    # Solve
    model.solve()

    return {
        "status": LpStatus[model.status],
        "recommended_order": int(order_qty.varValue or 0),
        "shortage": int(shortage.varValue or 0),
        "total_cost": round(value(model.objective), 2)
    }

In [26]:
# ==========================================
# Example Forecast
# ==========================================

predicted_demand = 420

result = optimize_inventory(
    predicted_demand=predicted_demand
)

print("\nOptimization Result:\n")

print(result)


Optimization Result:

{'status': 'Optimal', 'recommended_order': 420, 'shortage': 0, 'total_cost': 21840.0}


In [27]:
# ==========================================
# AI Critic / Refiner
# ==========================================

def critic_refiner(
    predicted_demand,
    optimization_result,
    insights=None
):

    issues = []

    # ==============================
    # Optimization Status Check
    # ==============================
    if optimization_result.get("status") != "Optimal":
        issues.append(
            f"Optimization warning: "
            f"{optimization_result.get('status', 'Unknown')}"
        )

    # ==============================
    # Extract Market Insight Text
    # ==============================
    insight_text = ""

    if insights:

        extracted_text = []

        for doc in insights:

            # Handle LangChain Document objects
            if hasattr(doc, "page_content"):
                extracted_text.append(
                    doc.page_content.lower()
                )

            # Handle plain strings
            elif isinstance(doc, str):
                extracted_text.append(
                    doc.lower()
                )

        insight_text = " ".join(extracted_text)

    # ==============================
    # Supplier Risk
    # ==============================
    if "blacklisted" in insight_text:
        issues.append(
            "Risk detected: Supplier issues found."
        )

    # ==============================
    # Freight Risk
    # ==============================
    if "freight" in insight_text:
        issues.append(
            "Logistics warning: Freight costs rising."
        )

    # ==============================
    # Overstock Risk
    # ==============================
    recommended_order = optimization_result.get(
        "recommended_order", 0
    )

    if recommended_order is None:
        recommended_order = 0

    if recommended_order > predicted_demand * 1.5:
        issues.append(
            "Overstock risk detected."
        )

    # ==============================
    # Final Decision
    # ==============================
    status = (
        "PASS"
        if len(issues) == 0
        else "REVISE"
    )

    # ==============================
    # Return Final Review
    # ==============================
    return {
        "status": status,
        "issues": issues,
        "market_context": insight_text,
        "optimization_status": optimization_result.get("status")
    }

In [29]:
# ==========================================
# Test Critic / Refiner
# ==========================================
result = {
    "status": "Optimal",
    "recommended_order": 500
}

insights = [
    "Supplier C blacklisted",
    "Freight costs rising"
]

review = critic_refiner(
    predicted_demand=420,
    optimization_result=result,
    insights=insights
)

print(review)

{'status': 'REVISE', 'issues': ['Risk detected: Supplier issues found.', 'Logistics warning: Freight costs rising.'], 'market_context': 'supplier c blacklisted freight costs rising', 'optimization_status': 'Optimal'}


In [31]:
def generate_procurement_report(sku, predicted_demand, optimization_result,
                                 critic_result, market_insights):
    lines = [
        "======================================",
        " PROCUREMENT EXECUTIVE REPORT",
        "======================================",
        f"SKU:                    {sku}",
        f"Predicted 4-Week Demand: {predicted_demand} units",
        f"Recommended Order:       {optimization_result['recommended_order']} units",
        f"Estimated Cost:          ${optimization_result['total_cost']}",
        f"Decision Status:         {critic_result['status']}",
        "\nTop Market Insights:"
    ]
    for i, insight in enumerate(market_insights, 1):
        lines.append(f"  {i}. {insight}")

    lines.append("\nRisk Flags:")
    if not critic_result["issues"]:
        lines.append("  No major procurement risks detected.")
    else:
        for issue in critic_result["issues"]:
            lines.append(f"  - {issue}")

    lines.append("\nFinal Recommendation:")
    if critic_result["status"] == "PASS":
        lines.append("  Proceed with procurement plan.")
    else:
        lines.append("  Revise procurement strategy before placing order.")

    lines.append("======================================")
    report = "\n".join(lines)
    print(report)
    return report  # now reusable!

In [33]:
sku = "ITEM_001"
market_query = "Which suppliers are risky?"

# Full automated pipeline
predicted_demand = predict_demand(sku)         # use AI forecast, not hardcoded
result = optimize_inventory(predicted_demand)
insights = retriever.invoke(market_query)       # fetch once, reuse everywhere
review = critic_refiner(predicted_demand, result, insights)
generate_procurement_report(sku, predicted_demand, result, review, insights)

 PROCUREMENT EXECUTIVE REPORT
SKU:                    ITEM_001
Predicted 4-Week Demand: 326.95 units
Recommended Order:       327 units
Estimated Cost:          $17004.0
Decision Status:         REVISE

Top Market Insights:
  1. page_content='Supplier B offers 8% volume discount above 500 units.'
  2. page_content='Supplier C has been blacklisted in winter months due to delays.'
  3. page_content='Sea freight rates from Shanghai up 30% use air for urgent orders.'

Risk Flags:
  - Risk detected: Supplier issues found.
  - Logistics warning: Freight costs rising.

Final Recommendation:
  Revise procurement strategy before placing order.


"======================================\n PROCUREMENT EXECUTIVE REPORT\n======================================\nSKU:                    ITEM_001\nPredicted 4-Week Demand: 326.95 units\nRecommended Order:       327 units\nEstimated Cost:          $17004.0\nDecision Status:         REVISE\n\nTop Market Insights:\n  1. page_content='Supplier B offers 8% volume discount above 500 units.'\n  2. page_content='Supplier C has been blacklisted in winter months due to delays.'\n  3. page_content='Sea freight rates from Shanghai up 30% use air for urgent orders.'\n\nRisk Flags:\n  - Risk detected: Supplier issues found.\n  - Logistics warning: Freight costs rising.\n\nFinal Recommendation:\n  Revise procurement strategy before placing order.\n======================================"

In [35]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("intellistock.db")

df = pd.read_sql("""
SELECT sku, AVG(sales) as avg_sales
FROM sales
GROUP BY sku
""", conn)

print(df)

conn.close()

        sku   avg_sales
0  ITEM_001   80.133758
1  ITEM_002  149.337580
2  ITEM_003  300.331210


In [37]:
 %pip install groq 

Note: you may need to restart the kernel to use updated packages.


In [39]:
from groq import Groq

client = Groq(api_key="")

# test it works
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print(response.choices[0].message.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [41]:
#LLM CRITIC
def llm_critic(sku, predicted_demand, order_qty, total_cost, market_context):
    
    prompt = f"""
You are a procurement manager reviewing an order plan.

SKU: {sku}
Predicted Demand: {predicted_demand} units
Recommended Order: {order_qty} units
Total Cost: {total_cost}
Market Context: {market_context}

Business Rules:
- Never order more than 600 units at once
- Total cost must not exceed 27000
- Flag any blacklisted suppliers
- Flag if order is more than 2x predicted demand

Review this plan and respond in this exact format:
VERDICT: PASS or REVISE
REASON: one sentence explanation
WARNING: any specific risk (or NONE)
"""
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    
    result = response.choices[0].message.content
    print(f"LLM Critic Response:\n{result}")
    return result

# Test it
market_ctx = "\n".join([d.page_content for d in 
                        retriever.invoke("supplier risks")])

llm_critic(
    sku="ITEM_001",
    predicted_demand=420,
    order_qty=462,
    total_cost=24024,
    market_context=market_ctx
)

LLM Critic Response:
VERDICT: REVISE
REASON: The recommended order of 462 units is within the allowed limits, but exploring a larger order to take advantage of Supplier B's 8% volume discount above 500 units could be beneficial, considering the current total cost is below the maximum allowed.


"VERDICT: REVISE\nREASON: The recommended order of 462 units is within the allowed limits, but exploring a larger order to take advantage of Supplier B's 8% volume discount above 500 units could be beneficial, considering the current total cost is below the maximum allowed.\nWARNING: NONE"

In [43]:
#LLM REPORT GENERATOR
def llm_generate_report(sku, predicted_demand, order_qty, 
                        total_cost, market_context, verdict):
    
    prompt = f"""
You are a supply chain analyst. Write a professional 
procurement report in 3 bullet points.

SKU: {sku}
Predicted 4-week demand: {predicted_demand} units
Recommended order: {order_qty} units
Total cost: {total_cost}
Market intelligence: {market_context}
Critic verdict: {verdict}

Format your response exactly like this:
- Demand insight: ...
- Procurement recommendation: ...
- Risk alert: ...
"""
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    
    report = response.choices[0].message.content
    print(f"AI Generated Report:\n{report}")
    return report

# Test it
llm_generate_report(
    sku="ITEM_001",
    predicted_demand=420,
    order_qty=462,
    total_cost=24024,
    market_context=market_ctx,
    verdict="REVISE"
)

AI Generated Report:
- Demand insight: Based on historical data and current market trends, the predicted 4-week demand for ITEM_001 is 420 units, with a recommended order quantity of 462 units to account for potential fluctuations and minimize stockouts.
- Procurement recommendation: Considering the total cost of $24,024 for the recommended order quantity, it may be beneficial to explore opportunities for volume discounts, such as those offered by Supplier B, which provides an 8% discount for orders above 500 units, potentially reducing costs and improving profitability.
- Risk alert: The procurement process should be cautious of supplier risks, including the increased lead time of Supplier A, which has risen from 3 to 6 weeks as of May 2025, and the blacklisting of Supplier C during winter months due to historical delays, highlighting the need for careful supplier selection and contingency planning to ensure uninterrupted supply and mitigate potential disruptions.


'- Demand insight: Based on historical data and current market trends, the predicted 4-week demand for ITEM_001 is 420 units, with a recommended order quantity of 462 units to account for potential fluctuations and minimize stockouts.\n- Procurement recommendation: Considering the total cost of $24,024 for the recommended order quantity, it may be beneficial to explore opportunities for volume discounts, such as those offered by Supplier B, which provides an 8% discount for orders above 500 units, potentially reducing costs and improving profitability.\n- Risk alert: The procurement process should be cautious of supplier risks, including the increased lead time of Supplier A, which has risen from 3 to 6 weeks as of May 2025, and the blacklisting of Supplier C during winter months due to historical delays, highlighting the need for careful supplier selection and contingency planning to ensure uninterrupted supply and mitigate potential disruptions.'

In [49]:
import json
import datetime

print("=" * 50)
print("INTELLISTOCK — FULL AI PIPELINE")
print("=" * 50)

sku = "ITEM_002"

# Phase 2 - Predict
print("\n📈 Phase 2: Predicting demand...")
predicted_demand = 420
print(f"Predicted demand: {predicted_demand} units")

# Phase 3 - RAG
print("\n🔍 Phase 3: Retrieving market context...")
market_ctx = "\n".join([d.page_content for d in
                        retriever.invoke("supplier risks shipping")])
print(f"Market context retrieved")

# Phase 4 - Optimize
# Phase 4 - Optimize
print("\n⚙️ Phase 4: Running LP optimizer...")

optimization_result = optimize_inventory(
    predicted_demand
)

order_qty = optimization_result[
    "recommended_order"
]

total_cost = optimization_result[
    "total_cost"
]

shortage = optimization_result[
    "shortage"
]

print(f"Recommended order: {order_qty} units")
print(f"Shortage          : {shortage} units")
print(f"Total cost        : ₹{total_cost}")

# Phase 5 - LLM Critic
print("\n🧠 Phase 5: LLM critic reviewing plan...")
critic_result = llm_critic(
    sku=sku,
    predicted_demand=predicted_demand,
    order_qty=order_qty,
    total_cost=total_cost,
    market_context=market_ctx
)

# Phase 6 - LLM Report
print("\n📋 Phase 6: Generating AI report...")
report_text = llm_generate_report(
    sku=sku,
    predicted_demand=predicted_demand,
    order_qty=order_qty,
    total_cost=total_cost,
    market_context=market_ctx,
    verdict=critic_result
)

# Save report
report = {
    "run_date": datetime.date.today().isoformat(),
    "sku": sku,
    "predicted_demand_4_weeks": predicted_demand,
    "recommended_order_qty": order_qty,
    "total_cost": total_cost,
    "critic_verdict": critic_result,
    "ai_report": report_text,
    "market_context": market_ctx
}

with open("procurement_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 50)
print("✅ PIPELINE COMPLETE!")
print(f"SKU            : {sku}")
print(f"Predicted demand: {predicted_demand} units")
print(f"Order quantity  : {order_qty} units")
print(f"Total cost      : ₹{total_cost}")
print("Report saved    : procurement_report.json")
print("=" * 50)

INTELLISTOCK — FULL AI PIPELINE

📈 Phase 2: Predicting demand...
Predicted demand: 420 units

🔍 Phase 3: Retrieving market context...
Market context retrieved

⚙️ Phase 4: Running LP optimizer...
Recommended order: 420 units
Shortage          : 0 units
Total cost        : ₹21840.0

🧠 Phase 5: LLM critic reviewing plan...
LLM Critic Response:
VERDICT: PASS
REASON: The recommended order of 420 units is within the allowed limits and does not exceed the total cost threshold of 27000.

📋 Phase 6: Generating AI report...
AI Generated Report:
- Demand insight: The predicted 4-week demand for ITEM_002 is 420 units, and based on this forecast, the recommended order quantity is also 420 units, with a total cost of 21840.0, which is below the threshold of 27000.
- Procurement recommendation: Considering the market intelligence, it is advisable to use air freight for urgent orders due to the 30% increase in sea freight rates from Shanghai, and to explore the 8% volume discount offered by Supplier 